# 04. Topic counts for Reddit posts

Frequency analysis and BERTopic told us what gets discussed. This notebook turns that into a number, by mapping the informal language people actually use to a formal topic and counting how many posts mention each one.

Women do not post about "dysmenorrhea", they post about bad cramps. The dictionary below is what bridges the two.

**Run from `notebooks/Reddit_Data/`.**

**Input:** `data/interim/cleaned_submissions.parquet` (from notebook 01)
**Output:** `data/processed/topic_counts.csv`, `data/interim/submissions_topic_analytics_long_form.parquet`

In [ ]:
import pandas as pd

df = pd.read_parquet("../../data/interim/cleaned_submissions.parquet")
df.info()

## Word mapping dictionary

One topic per entry, with every variant we saw people use for it. Brand names count, since posts say Mirena more often than they say hormonal IUD.

In [ ]:
TOPIC_KEYWORDS = {

    # --- Menstrual cycle & bleeding ---
    "menstrual_cycle": [
        "menstrual", "period", "periods", "menstruation", "cycle"
    ],
    "dysmenorrhea": [
        "dysmenorrhea", "bad periods", "cramps", "cramping", "bad cramps",
        "period pain", "menstrual pain"
    ],
    "heavy_bleeding": [
        "menorrhagia", "heavy bleeding", "heavy periods", "heavy period",
        "heavy flow", "flooding", "bleeding through", "soaking through",
        "a lot of blood", "blood everywhere", "blood everywhere through"
    ],
    "abnormal_bleeding": [
        "abnormal uterine bleeding", "irregular bleeding", "spotting",
        "breakthrough bleeding", "intermenstrual bleeding", "brown discharge",
        "bleeding between periods", "weird bleeding", "weird blood"
    ],
    "irregular_periods": [
        "irregular periods", "irregular period", "irregular cycle",
        "irregular cycles", "missed period", "late period", "skipped period",
        "period late", "late periods"
    ],
    "amenorrhea": [
        "amenorrhea", "no period", "period stopped", "lost my period",
        "missing periods", "no periods", "periods stopped", "lost my periods"
    ],
    "pms_pmdd": [
        "pms", "pmdd", "premenstrual syndrome", "premenstrual dysphoric disorder"
    ],

    # --- Contraception ---
    "oral_contraceptives": [
        "birth control pill", "birth control pills", "bc pill", "bcp",
        "the pill", "oral contraceptive", "combination pill",
        "combined pill", "mini pill", "minipill", "progestin only pill",
        "pop", "bc pills", "oral contraceptives"
    ],
    "iud": [
        "iud", "intrauterine device", "mirena", "kyleena", "skyla",
        "paragard", "copper", "liletta", "hormonal iud"
    ],
    "implant_shot_ring_patch": [
        "nexplanon", "the implant", "birth control implant", "depo",
        "depo shot", "depo-provera", "the shot", "nuvaring", "the ring",
        "birth control patch", "the patch", "xulane"
    ],
    "contraceptive_side_effects": [
        "bc side effects", "birth control side effects", "pill side effects",
        "hormonal side effects", "side effects from birth control", "side effects from bc"
    ],
    "tubal_ligation": [
        "tubal ligation", "tubes tied", "tubal ligations", "tubal ligated",
        "bilateral salpingectomy", "sterilization"
    ],

    # --- Vaginal / reproductive infections ---
    "yeast_infection": [
        "yeast infection", "yeast infections", "vulvovaginal candidiasis",
        "candida", "monistat", "diflucan", "fluconazole", "yeast infected"
    ],
    "bacterial_vaginosis": [
        "bacterial vaginosis", "bv", "metronidazole", "flagyl", "bac vag",
        "bacterial vag"
    ],
    "uti": [
        "uti", "utis", "urinary tract infection", "bladder infection",
        "burning when i pee"
    ],
    "boric_acid": [
        "boric acid"
    ],
    "vaginal_ph_microbiome": [
        "vaginal ph", "ph balance", "ph strips", "vaginal microbiome",
        "vaginal flora", "microbiome"
    ],
    "probiotics": [
        "probiotic", "probiotics", "prebiotic", "rephresh", "vh essentials"
    ],

    # --- Pelvic pain & structural conditions ---
    "pelvic_pain": [
        "pelvic pain", "lower abdomen pain", "abdominal pain", "lower abdominal pain"
    ],
    "ovarian_cyst": [
        "ovarian cyst", "ovarian cysts", "cyst"
    ],
    "ovarian_torsion": [
        "ovarian torsion", "torsion", "twisted ovary"
    ],
    "endometriosis": [
        "endometriosis", "endo", "endometrioma", "endometriomas"
    ],
    "adenomyosis": [
        "adenomyosis"
    ],
    "pelvic_floor": [
        "pelvic floor", "pelvic floor therapy", "pelvic floor pt",
        "pelvic floor physical therapy", "pelvic floor dysfunction",
        "pelvic floor hypertonicity"
    ],
    "vulvodynia": [
        "vulvodynia"
    ],
    "interstitial_cystitis": [
        "interstitial cystitis", "ic", "painful bladder syndrome"
    ],
    "bartholin_cyst": [
        "bartholin cyst", "bartholin's cyst", "bartholin gland cyst"
    ],

    # --- Breast health ---
    "breast_lump": [
        "breast lump", "lump in my breast", "breast lumps", "palpable lump"
    ],
    "fibroadenoma": [
        "fibroadenoma", "fibroadenomas"
    ],
    "breast_pain": [
        "breast pain", "mastalgia", "sore breasts", "breast tenderness",
        "nipple pain"
    ],
    "breast_cancer_screening": [
        "mammogram", "mammography", "breast ultrasound", "breast cancer",
        "birads", "bi-rads"
    ],

    # --- Hormonal / endocrine ---
    "pcos": [
        "pcos", "polycystic ovary syndrome", "polycystic ovarian syndrome"
    ],
    "hormonal_acne": [
        "hormonal acne", "hormonal breakouts", "chin acne", "jawline acne"
    ],
    "hirsutism": [
        "hirsutism", "excess facial hair", "chin hairs", "facial hair growth"
    ],
    "thyroid": [
        "thyroid", "tsh", "hypothyroidism", "hyperthyroidism",
        "hashimoto's", "hashimotos", "levothyroxine", "synthroid",
        "thyroid peroxidase"
    ],
    "menopause": [
        "menopause", "perimenopause", "menopausal", "hrt",
        "hormone replacement therapy", "hormone therapy"
    ],
    "hot_flashes_night_sweats": [
        "hot flashes", "hot flash", "night sweats", "night sweat"
    ],

    # --- Sexual health ---
    "sti_std": [
        "sti", "std", "stis", "stds", "chlamydia", "gonorrhea", "herpes",
        "hpv", "sexually transmitted infection", "sexually transmitted disease"
    ],
    "painful_sex": [
        "painful sex", "dyspareunia", "sex hurts", "pain during sex",
        "hurts during sex"
    ],
    "libido": [
        "libido", "sex drive", "low libido", "no sex drive", "sexual desire"
    ],
    "unprotected_sex": [
        "unprotected sex", "no condom", "without a condom", "pull out method",
        "pullout method"
    ],

    # --- Pregnancy-adjacent ---
    "pregnancy_test": [
        "pregnancy test", "pregnancy tests", "took a test", "positive test",
        "negative test", "hpt"
    ],
    "abortion": [
        "abortion", "medical abortion", "misoprostol", "mifepristone",
        "abortion pill", "plan b", "morning after pill" # might want to remove plan b and morning after pill because it's technically different?
    ],
    "abortion_policy": [
        "roe v wade", "roe vs wade", "abortion rights", "abortion ban",
        "abortion access", "women's health protection act"
    ],

    # --- Body & general symptoms ---
    "hair_loss": [
        "hair loss", "alopecia", "hair thinning", "hair falling out",
        "thinning hair", "losing hair"
    ],
    "iron_anemia": [
        "iron deficiency", "anemia", "anemic", "ferritin", "low iron",
        "iron infusion", "iron supplements"
    ],
    "bloating": [
        "bloating", "bloated", "bloat", "gassy", "gas"
    ],
    "hemorrhoids": [
        "hemorrhoid", "hemorrhoids", "external hemorrhoid", "internal hemorrhoid"
    ],
    "nausea": [
        "nausea", "nauseous", "nauseated", "throwing up", "vomiting"
    ],
    "fatigue_sleep": [
        "fatigue", "exhausted", "exhaustion", "tired all the time",
        "insomnia", "can't sleep", "cant sleep", "sleep issues"
    ],
    "heart_palpitations": [
        "heart palpitations", "palpitations", "heart racing", "racing heart",
        "high heart rate", "irregular heartbeat"
    ],
    "headache_migraine": [
        "headache", "headaches", "migraine", "migraines", "tension headache"
    ],
    "allergic_reaction": [
        "hives", "allergic reaction", "rash", "itchy rash", "allergy"
    ],

    # --- Hygiene & products ---
    "tampon_safety": [
        "tampon", "tampons", "toxic shock syndrome", "tss"
    ],
    "menstrual_cup": [
        "menstrual cup", "period cup", "diva cup", "menstrual disc"
    ],
    "period_tracking_apps": [
        "period tracker", "period tracking app", "cycle tracker", "flo",
        "natural cycles", "clue"
    ],

    # --- Vaccines & broader health ---
    "covid_vaccine": [
        "covid vaccine", "covid-19 vaccine", "pfizer", "moderna",
        "covid shot", "vaccine and period", "vaccine side effects"
    ],
}

## Topic patterns

One regex per topic that matches any of its variants. Word boundaries keep short terms from matching inside longer words, so "bc" does not fire on "back". Multi-word phrases are matched literally and sorted longest first, so a phrase is not shadowed by a shorter term inside it.

In [ ]:
import re

def build_topic_patterns(topic_keywords: dict) -> dict:
    patterns = {}
    for topic, terms in topic_keywords.items():
        terms_sorted = sorted(set(terms), key=len, reverse=True)
        escaped = [re.escape(t) for t in terms_sorted]
        pattern = r"\b(?:" + "|".join(escaped) + r")\b"
        patterns[topic] = re.compile(pattern, flags=re.IGNORECASE)
    return patterns

TOPIC_PATTERNS = build_topic_patterns(TOPIC_KEYWORDS)
print(f"{len(TOPIC_PATTERNS)} topic patterns built")

## Counting function

For each topic, counts how many posts mention any of its keywords in either the title or the body. Also adds one boolean column per topic to `df` for filtering later.

In [ ]:
def count_topic_mentions(df: pd.DataFrame,
                         title_col: str = "title",
                         text_col: str = "selftext") -> pd.DataFrame:
    # fillna avoids errors on missing text
    combined = (
        df[title_col].fillna("").astype(str)
        + " "
        + df[text_col].fillna("").astype(str)
    )

    results = []
    for topic, pattern in TOPIC_PATTERNS.items():
        matches = combined.str.contains(pattern)
        df[f"topic__{topic}"] = matches
        results.append({"topic": topic, "post_count": int(matches.sum())})

    return (pd.DataFrame(results)
            .sort_values("post_count", ascending=False)
            .reset_index(drop=True))

In [ ]:
counts_df = count_topic_mentions(df)
counts_df

In [ ]:
counts_df.to_csv("../../data/processed/topic_counts.csv", index=False)

## Tag each post with its topics

Now we pair each post with the topics it contains so we can analyze by topic.

- A post about one thing gets a list with one topic
- A post covering several things gets a list with several
- A post matching nothing gets an empty list

In [ ]:
def tag_post_topics(df: pd.DataFrame,
                    title_col: str = "title",
                    text_col: str = "selftext",
                    new_col: str = "matched_topics") -> pd.DataFrame:
    combined = (
        df[title_col].fillna("").astype(str)
        + " "
        + df[text_col].fillna("").astype(str)
    )

    def find_topics(text: str) -> list:
        return [topic for topic, pattern in TOPIC_PATTERNS.items() if pattern.search(text)]

    df[new_col] = combined.apply(find_topics)
    return df

df = tag_post_topics(df)

In [ ]:
# How much of the corpus our topics actually cover
unmatched = (df["matched_topics"].str.len() == 0).sum()
print(f"{unmatched} of {len(df)} posts matched no topic ({unmatched / len(df):.1%})")

## Trim to analysis columns

Everything from here on is counts and metadata, so the token and keyword list columns can go.

In [ ]:
token_cols = [c for c in df.columns
              if c.endswith(("_tokens", "_keywords", "_bigrams"))]
df = df.drop(columns=token_cols)
df = df.dropna(axis=1, how="all")
df.info()

## Long form

One row per post per topic. This is the shape the database and the topic hierarchy both expect.

In [ ]:
long_df = df.explode("matched_topics")

# Should match the counts from earlier
long_df["matched_topics"].value_counts().head(20)

In [ ]:
long_df.to_parquet("../../data/interim/submissions_topic_analytics_long_form.parquet")
print(f"{len(long_df)} post-topic rows saved")